# FinSight AI — Research Notebook 02: Feature Engineering & Leakage Prevention
## Constructing Rolling Indicators, Lags, and Enforcing Chronological Train/Test Split Boundaries

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "backend"))  # run from the ml_research/ folder
import numpy as np
import pandas as pd
from app.data import market_data as md
from app.ml.feature_engineering import FEATURE_COLUMNS, compute_features, create_target, time_series_split

In [ ]:
# Ingest OHLCV
raw, is_demo = md.history("AAPL", "5y", "1d")
print(f"Raw observations: {len(raw)} (simulated: {is_demo})")

In [ ]:
# Scale-invariant technical features (ratios, oscillators, returns)
feat_df = compute_features(raw)
print(f"Engineered features: {len(FEATURE_COLUMNS)}  ·  usable rows after warm-up: {len(feat_df)}")
feat_df[FEATURE_COLUMNS].describe().T.round(4).head(12)

In [ ]:
# Target = forward 5-day return. Purged chronological split (gap = horizon) prevents label leakage.
H = 5
X, y = create_target(feat_df, horizon=H)
X_train, X_val, X_test, y_train, y_val, y_test = time_series_split(X, y, gap=H)
for name, part in [("Train", X_train), ("Validation", X_val), ("Test", X_test)]:
    print(f"{name:<11} {part.index[0].date()} → {part.index[-1].date()} ({len(part)} rows)")
assert X_train.index[-1] < X_val.index[0] < X_test.index[0], "Time overlap detected!"

In [ ]:
# Look-ahead check: features computed on a truncated history must equal the full-history features
past = compute_features(raw.iloc[:-60])
full = compute_features(raw)
diff = (past[FEATURE_COLUMNS] - full.loc[past.index, FEATURE_COLUMNS]).abs().max().max()
print(f"Max difference on shared rows: {diff:.2e}  →  {'no look-ahead ✓' if diff < 1e-9 else 'LEAK!'}")